### Import required libraries

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType

In [0]:
%run /Workspace/Shopvista/Shopvista-Data-Engineering-Project-_databricks/1_setup/utilities

In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
dbutils.widgets.text("catalog","abc","catalog")
dbutils.widgets.text("data_source","xyz", "data_source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source=dbutils.widgets.get("data_source")

print(catalog, data_source)

In [0]:
# Define schema for product table

product_schema = StructType([
    StructField("product_id", StringType(), False),
    StructField("Sku", StringType(), True),
    StructField("category_code", StringType(), True),
    StructField("brand_code", StringType(), True),
    StructField("color", StringType(), True),
    StructField("size", StringType(), True),
    StructField("material", StringType(), True),
    StructField("weight_grams", StringType(), True), #datatype is string due to incoming data contain anamolies
    StructField("length_cm", StringType(), True), #datatype is string due to incoming data contain anamolies
    StructField("width_cm", FloatType(), True),
    StructField("height_cm", FloatType(), True),
    StructField("rating_count", IntegerType(), True)
])

In [0]:
#  show the path to the datasource

base_path = f"s3://shopvista-sv/{data_source}/*csv"

print(base_path)

In [0]:

# load data into dataframe and add metadata
df = (
    spark.read.format("csv")
        .option("header", True)
        .schema(product_schema)
        .load(base_path)
        .withColumn("data_source", F.current_timestamp()) # add metadata columns
        .select("*", "_metadata.file_name", "_metadata.file_path") # add metadata columns
)

In [0]:
display(df.limit(5))

### write product data to bronze schema

In [0]:
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")